<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# เทรนโมเดลตรวจจับไฟจากภาพถ่ายทางอากาศ (YOLO26)

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🇹🇭 **ภาษาไทย** (เอกสารฉบับนี้) · [🇬🇧 English](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/drone_fire_detection_yolo26.ipynb)

เทรนโมเดลตรวจจับไฟบนภาพถ่ายทางอากาศจากอากาศยานไร้คนขับ ด้วย
[Ultralytics YOLO26](https://docs.ultralytics.com/models/yolo26/)

**Runtime:** `Runtime` → `Change runtime type` → **T4 GPU** (หรือดีกว่านั้น) แล้วกด `Save`

พอเทรนเสร็จ ให้นำไฟล์ `best.pt` ที่ได้ไปรันต่อใน [`Supervision_Image_Inferencing.ipynb`](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/Supervision_Image_Inferencing.ipynb)
และ [`Supervision_Video_Inferencing.ipynb`](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/th/Supervision_Video_Inferencing.ipynb) กับภาพนิ่งและวิดีโอ

## 1. ตรวจสอบ GPU

In [1]:
!nvidia-smi

Wed Aug 19 15:56:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. ติดตั้งไลบรารี

กำหนดเป็นเวอร์ชันขั้นต่ำ ไม่ได้ตรึงเวอร์ชันแบบเป๊ะ ๆ เพราะแค่ต้องการการันตีว่า API
ที่โน้ตบุ๊กนี้เรียกใช้มีอยู่จริง แล้วปล่อยให้ pip ไปจับคู่กับ PyTorch รุ่นที่ Colab ติดตั้งมาให้เอง

In [2]:
%pip install -q "ultralytics>=8.4.122" "supervision>=0.30.0"

import ultralytics
ultralytics.checks()

Ultralytics 8.4.123 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.2/112.6 GB disk)


In [3]:
import os
from pathlib import Path

from ultralytics import YOLO

HOME = Path.cwd()
print("HOME:", HOME)

HOME: /content


## 3. ดาวน์โหลดชุดข้อมูลจาก Roboflow

ต้องใช้ Roboflow API key แบบไม่มีค่าใช้จ่าย (<https://app.roboflow.com/settings/api>)

ถ้ารันบน Colab ให้เก็บ key ไว้ครั้งเดียวในแผง 🔑 **Secrets** ทางแถบซ้ายมือ ตั้งชื่อว่า
`ROBOFLOW_API_KEY` แล้วเปิดสิทธิ์ *Notebook access* เซลล์ด้านล่างจะอ่านจากที่นั่นก่อน
ถ้าไม่เจอจึงไปอ่านจากตัวแปรสภาพแวดล้อม แล้วค่อยถามผู้ใช้เป็นทางเลือกสุดท้าย
วิธีนี้ทำให้ key ไม่ติดไปกับตัวโน้ตบุ๊กตอน commit

In [4]:
%pip install -q "roboflow>=1.4.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 122.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.8 MB/s eta 0:00:00


In [6]:
import getpass
import os


def get_roboflow_api_key() -> str:
    """อ่าน API key จาก Colab Secrets ก่อน ถ้าไม่เจอจึงไปอ่านจากตัวแปรสภาพแวดล้อม แล้วค่อยถามผู้ใช้"""
    try:
        from google.colab import userdata  # type: ignore

        key = userdata.get("ROBOFLOW_API_KEY")
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get("ROBOFLOW_API_KEY")
    if key:
        return key
    return getpass.getpass("Roboflow API key: ")


ROBOFLOW_API_KEY = get_roboflow_api_key()

In [ ]:
from roboflow import Roboflow

DATASETS_DIR = HOME / "datasets"
DATASETS_DIR.mkdir(exist_ok=True)
os.chdir(DATASETS_DIR)

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("tim-4ijf0").project("drone-fire-detection-byija")

# "yolov8" คือชื่อ *รูปแบบการจัดวางไฟล์ตอนส่งออก* ของชุดข้อมูล (ภาพ + ไฟล์ label .txt ตามมาตรฐาน
# YOLO + data.yaml) ไม่ใช่เวอร์ชันของโมเดล ฝั่ง Roboflow ไม่มีรูปแบบส่งออกชื่อ "yolo26" และไม่จำเป็นต้องมี
# เพราะนี่คือรูปแบบ YOLO มาตรฐานที่ YOLO26 เอาไปเทรนต่อได้เลยโดยไม่ต้องแก้อะไร ส่วน data.yaml
# ที่ส่งออกมาจะใช้พาธแบบสัมพัทธ์ "../train/images" ซึ่ง Ultralytics จะไล่หาโดยอิงจากโฟลเดอร์ของ
# data.yaml เอง เราจึงไม่ต้องไปยุ่งกับพาธเพิ่ม
dataset = project.version(1).download("yolov8")

os.chdir(HOME)

DATA_YAML = Path(dataset.location) / "data.yaml"
print("data.yaml:", DATA_YAML)
print(DATA_YAML.read_text())

## 4. เทรนโมเดล

YOLO26 เป็นสถาปัตยกรรมแบบครบวงจร (end-to-end) และไม่ผ่าน NMS จึงไม่มีค่าขีดแบ่ง `iou`
ของ NMS ให้ต้องปรับ เพราะโมเดลคายกรอบสุดท้ายออกมาให้เลย

โค้ดเก็บค่า `results.save_dir` ไว้ เพื่อให้เซลล์ถัด ๆ ไปไม่ต้อง hardcode พาธ `runs/detect/train`
เนื่องจาก Ultralytics จะไล่เลขเป็น `train2`, `train3`, … ทุกครั้งที่รันซ้ำ

ถ้าเจอปัญหาหน่วยความจำไม่พอ (out of memory) ให้ลด `imgsz` เหลือ 640 หรือเปลี่ยน `model` เป็น `yolo26n.pt`

In [ ]:
MODEL_ARCH = "yolo26m.pt"  # n / s / m / l / x

model = YOLO(MODEL_ARCH)

train_results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=800,
    plots=True,
)

RUN_DIR = Path(train_results.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
print("run dir:", RUN_DIR)
print("best weights:", BEST_WEIGHTS)

In [ ]:
from IPython.display import Image, display

for artifact in ["confusion_matrix.png", "results.png", "val_batch0_pred.jpg"]:
    path = RUN_DIR / artifact
    if path.exists():
        print(artifact)
        display(Image(filename=str(path), width=700))
    else:
        print("ไม่พบไฟล์:", artifact)

## 5. ตรวจสอบความแม่นยำของโมเดล (validate)

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
metrics = best_model.val(data=str(DATA_YAML))

print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP75:    {metrics.box.map75:.4f}")

## 6. ลองทำนายผลบนชุดทดสอบ (test split)

In [ ]:
predict_results = best_model.predict(
    source=str(Path(dataset.location) / "test" / "images"),
    conf=0.25,
    save=True,
)

PREDICT_DIR = Path(predict_results[0].save_dir)
print("predictions:", PREDICT_DIR)

In [ ]:
import glob

for image_path in sorted(glob.glob(f"{PREDICT_DIR}/*.jpg"))[:3]:
    display(Image(filename=image_path, width=700))

## 7. ส่งออกไฟล์ weights

ดาวน์โหลด `best.pt` เก็บไว้ เพื่อเอาไปป้อนให้โน้ตบุ๊กฝั่ง inference ทั้งสองไฟล์

In [ ]:
try:
    from google.colab import files  # type: ignore

    files.download(str(BEST_WEIGHTS))
except ImportError:
    print("ไม่ได้รันบน Colab ไฟล์ weights อยู่ที่:", BEST_WEIGHTS)